# style-lora Colab quickstart (한 번에 끝내기)

런타임 → 런타임 유형 변경 → GPU(A100 권장, 없으면 T4)로 설정한 뒤,
아래 셀을 **위에서부터 차례대로 실행**하면 끝납니다.
중간에 문단을 고르거나 파일을 나누는 작업은 없습니다 — 데이터 분리와 평가 세트 구성은
자동으로 처리됩니다.

**절대 이 txt 원문을 공개 저장소에 커밋하지 마세요.** 이 노트북은 Colab 세션에만 파일을 올립니다.

In [ ]:
from getpass import getpass
token = getpass('GitHub 토큰 입력 (저장소가 private이라 필요): ')
!rm -rf repo
!git clone -b claude/novelai-custom-module-limits-a4uncu https://{token}@github.com/yeou77/-.git repo
%cd repo/style-lora
!pip install -q -r requirements.txt

## 1) 소설 통째로 업로드
정제된 본문만 있는 txt 2개를 그대로 올립니다 (작가의 말/후기 제거된 상태). 나눠서 올릴 필요 없습니다.

In [ ]:
from google.colab import files
uploaded = files.upload()  # 소설 txt 2개 그대로 선택
import shutil, os
os.makedirs('data/raw', exist_ok=True)
for name in uploaded:
    shutil.move(name, f'data/raw/{name}')
print(os.listdir('data/raw'))

## 2) 자동 전처리 (분리 + 평가셋 무작위 분리, 모두 한 번에)
평가용으로 떼 문단은 직접 고르지 않고 무작위로 떼어집니다 — 더 정교한 결과를 원하면
`scripts/preprocess.py candidates` / `build`로 직접 고르는 버전을 대신 쓰세요 (README 참고).

In [ ]:
!python scripts/preprocess.py auto

## 3) QLoRA 학습 (여기가 시간이 걸리는 단계)

In [ ]:
!python scripts/train_lora.py \
    --model_name beomi/Llama-3-Open-Ko-8B-Instruct-preview \
    --train_file data/processed/train_corpus.txt \
    --output_dir outputs/style-lora

## 4) base vs +lora 비교 리포트

In [ ]:
!python scripts/eval.py --lora_dir outputs/style-lora --out outputs/eval_report.md
print(open('outputs/eval_report.md', encoding='utf-8').read())

## 5) 결과 저장 (중요: LoRA 가중치는 Colab 세션이 끝나면 사라집니다)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r outputs/style-lora /content/drive/MyDrive/style-lora